**Transform Orders Data-String To JSON Object**

In [0]:
SELECT *
FROM gizmobox_gr.bronze.v_orders

**Pre-Process JSON String To Fix The Data Quality Issues**

In [0]:
SELECT value, regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": \1') As fixed_value
FROM gizmobox_gr.bronze.v_orders

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_orders AS
SELECT value, regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": \1') As fixed_value
FROM gizmobox_gr.bronze.v_orders

**Transform JSON String To JSON Object**

In [0]:
SELECT schema_of_json(fixed_value) AS schema
FROM tv_orders

In [0]:
SELECT from_json(fixed_value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_schema
FROM tv_orders

**Write Transformed Data To Silver Schema**

In [0]:
CREATE OR REPLACE TABLE gizmobox_gr.silver.orders_json
AS
SELECT from_json(fixed_value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value
FROM tv_orders

In [0]:
SELECT * FROM gizmobox_gr.silver.orders_json;